# HW3 · 01 — EDA and fixed validation split
Goal: understand the data (class balance by language/domain, lengths, examples, what makes *fusion* different)
and define ONE fixed train/validation split that every other notebook re-creates with the same code.

In [ ]:
import glob, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 200)
SEED = 42

In [ ]:
# Find the competition files wherever Kaggle mounted them
DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print(DATA_DIR)
print("train", train.shape, "test", test.shape)
print("train columns:", list(train.columns))
print("test columns: ", list(test.columns))
train.head(3)

In [ ]:
# Column names (auto-detected; fix by hand if the print above shows other names)
TEXT_COL, LABEL_COL = "text", "label"
LANG_COL = next((c for c in ["language", "lang"] if c in train.columns), None)
DOMAIN_COL = next((c for c in ["domain", "source", "genre"] if c in train.columns), None)
print("lang column:", LANG_COL, "| domain column:", DOMAIN_COL)

## 1. Class balance

In [ ]:
print(train[LABEL_COL].value_counts(normalize=True).round(3))
for col in [LANG_COL, DOMAIN_COL]:
    if col:
        print(f"\n--- label share by {col} (rows = {col}) ---")
        ct = pd.crosstab(train[col], train[LABEL_COL], margins=True)
        display(ct)
        display(pd.crosstab(train[col], train[LABEL_COL], normalize="index").round(3))

In [ ]:
if LANG_COL and DOMAIN_COL:
    display(pd.crosstab([train[LANG_COL], train[DOMAIN_COL]], train[LABEL_COL]))

In [ ]:
# Does test look like train? (only feature distributions, no labels involved)
for col in [LANG_COL, DOMAIN_COL]:
    if col and col in test.columns:
        cmp = pd.concat([train[col].value_counts(normalize=True).rename("train"),
                         test[col].value_counts(normalize=True).rename("test")], axis=1).round(3)
        display(cmp)

## 2. Text lengths

In [ ]:
for df in (train, test):
    df["n_chars"] = df[TEXT_COL].str.len()
    df["n_words"] = df[TEXT_COL].str.split().str.len()

display(train.groupby(LABEL_COL)[["n_chars", "n_words"]].describe().round(0))
if DOMAIN_COL:
    display(train.pivot_table(index=DOMAIN_COL, columns=LABEL_COL, values="n_words", aggfunc="median"))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for lab, g in train.groupby(LABEL_COL):
    ax[0].hist(np.log10(g["n_words"].clip(lower=1)), bins=60, alpha=0.5, label=lab, density=True)
ax[0].set_xlabel("log10(#words)"); ax[0].legend(); ax[0].set_title("Length by class")
ax[1].hist(np.log10(test["n_words"].clip(lower=1)), bins=60, alpha=0.6, color="gray", density=True)
ax[1].set_xlabel("log10(#words)"); ax[1].set_title("Test length")
plt.tight_layout(); plt.savefig("eda_lengths.png", dpi=120); plt.show()

# rough token estimate: XLM-R uses ~1.3–1.6 tokens per word for EN/DE
for lim in (256, 512):
    share = (train["n_words"] * 1.4 > lim).mean()
    print(f"~share of train texts longer than {lim} tokens: {share:.2%}")

## 3. Examples of each class

In [ ]:
for lab in train[LABEL_COL].unique():
    print(f"================ {lab} ================")
    for t in train[train[LABEL_COL] == lab].sample(3, random_state=SEED)[TEXT_COL]:
        print("-", t[:400].replace("\n", " "), "...\n")

## 4. What makes fusion different? Simple stylometric statistics per class

In [ ]:
def stylo(text):
    words = text.split()
    sents = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    n_w = max(len(words), 1)
    sent_lens = [len(s.split()) for s in sents] or [0]
    return {
        "avg_word_len": np.mean([len(w) for w in words]) if words else 0,
        "avg_sent_len": np.mean(sent_lens),
        "std_sent_len": np.std(sent_lens),
        "type_token_ratio": len(set(w.lower() for w in words)) / n_w,
        "comma_per_word": text.count(",") / n_w,
        "punct_per_char": sum(not c.isalnum() and not c.isspace() for c in text) / max(len(text), 1),
        "newline_per_word": text.count("\n") / n_w,
        "digit_per_char": sum(c.isdigit() for c in text) / max(len(text), 1),
        "upper_per_char": sum(c.isupper() for c in text) / max(len(text), 1),
    }

sample = train.groupby(LABEL_COL).sample(n=3000, random_state=SEED)
feats = pd.DataFrame([stylo(t) for t in sample[TEXT_COL]], index=sample.index)
feats[LABEL_COL] = sample[LABEL_COL].values
display(feats.groupby(LABEL_COL).median().T.round(3))

# TTR depends on length, so compare it on fixed-size windows (first 200 words)
sample["ttr200"] = sample[TEXT_COL].map(lambda t: len(set(w.lower() for w in t.split()[:200])) / max(min(len(t.split()), 200), 1))
display(sample[sample["n_words"] >= 200].groupby(LABEL_COL)["ttr200"].median().round(3))

## 5. Duplicates / data quality

In [ ]:
print("duplicate texts in train:", train[TEXT_COL].duplicated().sum())
dup = train[train[TEXT_COL].duplicated(keep=False)]
if len(dup):
    print("duplicates with conflicting labels:", (dup.groupby(TEXT_COL)[LABEL_COL].nunique() > 1).sum())
print("train/test exact overlap:", test[TEXT_COL].isin(set(train[TEXT_COL])).sum())
print("empty / very short texts (<5 words):", (train["n_words"] < 5).sum())

## 6. Fixed split (same function is copied into every notebook)
* `val`: 10,000 rows, stratified by label × language — used for ALL model comparisons
* `train_sub`: 40,000 rows from the remaining data, same stratification — used to train every model

In [ ]:
def make_split(train, label_col="label", lang_col=None, n_val=10_000, n_train=40_000, seed=42):
    strat = train[label_col].astype(str) + ("_" + train[lang_col].astype(str) if lang_col else "")
    rest, val = train_test_split(train, test_size=n_val, stratify=strat, random_state=seed)
    strat_rest = strat.loc[rest.index]
    train_sub, _ = train_test_split(rest, train_size=n_train, stratify=strat_rest, random_state=seed)
    return train_sub.reset_index(drop=True), val.reset_index(drop=True)

train_sub, val = make_split(train, LABEL_COL, LANG_COL)
print(train_sub.shape, val.shape)
print(val[LABEL_COL].value_counts(normalize=True).round(3))
pd.concat([train_sub[["id"]].assign(split="train_sub"), val[["id"]].assign(split="val")]).to_csv("split_ids.csv", index=False)
print("first val ids (must match in every notebook):", val["id"].head(3).tolist())